In [1]:
!pip install langchain
!pip install openai
!pip install PyPDF2
!pip install faiss-cpu
!pip install tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 23.0 MB/s eta 0:00:00


In [2]:
!pip install langchain-huggingface sentence-transformers faiss-cpu pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.5/329.5 kB 8.1 MB/s eta 0:00:00


In [3]:
!pip install langchain-text-splitters

In [4]:
!pip install langchain_community
from PyPDF2 import PdfReader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import FAISS

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [5]:
embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
pdfreader=PdfReader('/sma unit-1.pdf')

In [8]:
from IPython.core.interactiveshell import page
from typing_extensions import Concatenate
raw_text=' '
for i,page in enumerate(pdfreader.pages):
  content=page.extract_text()
  if content:
    raw_text+=content

In [9]:
raw_text

' Social Media Analytics\nUnit -1\nJAWAHARLAL NEHRU TECHNOLOGICAL UNIVERSITY,HYDERABAD\nUNIVERSITY COLLEGE OF ENGINEERING,SCIENCE & TECHNOLOGY,HYDERABAD\nDEPARTMENT OF INFORMATION TECHNOLOGYTopics\n-World Wide Web\n-Web 1.0\n-Web 2.0\n-Web 3.0\n-Social Media\n-Core Characteristics of Social Media\n-Types of Social Media-Social Networking Sites\n-Using Facebook for Business Purposes\n-Content CommunitiesWorld Wide Web\nDefinition :The World Wide Web (WWW), commonly called “the web,” isasystem of\ninterlinked hypertext documents accessible viatheInternet through web browsers .\n•While theInternet provides theinfrastructure (network ofdevices, cables, satellites,\nrouters, and protocols), theWWW provides content and services intheform of\nwebsites, hyperlinks, andapplications .\n•Proposed byTim Berners -Lee in1989–1990 atCERN (European Organization for\nNuclear Research) .\n•Internet :Hardware andprotocols (TCP/IP) thatconnect computers .\n•WWW :Aservice running ontopoftheInternet, using 

In [10]:
text_splitter = CharacterTextSplitter(
    separator = "\n",
    chunk_size = 400,
    chunk_overlap  = 100,
    length_function = len,
)
texts=text_splitter.split_text(raw_text)

In [11]:
len(texts)

83

In [12]:
document_search=FAISS.from_texts(texts,embeddings)

In [13]:
document_search

In [14]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain_huggingface import HuggingFacePipeline # Changed to HuggingFacePipeline
from transformers import pipeline # Import pipeline
from google.colab import userdata
from operator import itemgetter

# Initialize LLM with a local Hugging Face pipeline
repo_id="google/flan-t5-base" # Using a smaller model for local execution

# Create a Hugging Face pipeline for text generation
pipe = pipeline(
    "text2text-generation",
    model=repo_id,
    max_new_tokens=50, # Limit output length to prevent very long responses
    temperature=0.5,
    device=0 # Use GPU if available, else CPU (-1)
)

# Wrap the pipeline in LangChain's HuggingFacePipeline
llm = HuggingFacePipeline(pipeline=pipe)

# Define the prompt template
prompt = ChatPromptTemplate.from_template(
    """Answer the question based only on the following context:\n\n{context}\n\nQuestion: {input}"""
)

# Assuming 'document_search' (FAISS vectorstore) is already initialized from previous cells
retriever = document_search.as_retriever()

# Define a function to format retrieved documents for the prompt
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Construct the RAG chain using LCEL
retrieval_chain = (
    RunnableParallel(
        context=itemgetter("input") | retriever | format_docs,
        input=itemgetter("input")
    )
    | prompt
    | llm
    | StrOutputParser()
)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cpu
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [15]:
chain = retrieval_chain

In [19]:
query = "Key Characteristics of web 1.0"
response = chain.invoke({"input": query})
print(response)

Token indices sequence length is longer than the specified maximum sequence length for this model (520 > 512). Running this sequence through the model will result in indexing errors


1)Static Web Pages 2)Read -Only Content 3)One -Way Communication 4)Limited User Contribution 5)Page Design and Layout 6)Navigation 7)File -
